In [3]:
import numpy as np
import pymysql
import pandas as pd
# ==================== 1. 범용 경로 설정 ====================
import sys
import os
from pathlib import Path

def setup_universal_paths():
    """
    어떤 PC에서도 작동하는 범용 경로 설정
    DATA 폴더를 자동으로 찾아 경로 추가
    """
    current = Path.cwd()

    # 상위 폴더를 탐색하며 DATA 폴더 찾기
    for parent in [current, *current.parents]:
        data_folder = parent / "DATA"
        if data_folder.exists():
            # 프로젝트 루트와 DATA 폴더 모두 추가
            if str(parent) not in sys.path:
                sys.path.insert(0, str(parent))
            if str(data_folder) not in sys.path:
                sys.path.insert(0, str(data_folder))

            print("=" * 70)
            print("📁 경로 설정 완료")
            print("=" * 70)
            print(f"✓ 프로젝트 루트: {parent}")
            print(f"✓ DATA 폴더:    {data_folder}")
            print(f"✓ 현재 위치:     {current}")
            print(f"✓ 운영체제:      {os.name}")
            print("=" * 70 + "\n")

            return {
                'project_root': parent,
                'data_folder': data_folder,
                'current': current
            }

    # 못 찾으면 에러
    raise FileNotFoundError(
        f"❌ DATA 폴더를 찾을 수 없습니다.\n"
        f"현재 위치: {current}\n"
        f"상위 폴더에 DATA 폴더가 있는지 확인하세요."
    )

# 경로 설정 실행
try:
    paths = setup_universal_paths()
except FileNotFoundError as e:
    print(e)
    print("\n대안: 수동으로 경로를 설정하세요.")
    # sys.path.insert(0, "여기에_프로젝트_루트_경로_입력")
    sys.exit(1)

📁 경로 설정 완료
✓ 프로젝트 루트: C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy
✓ DATA 폴더:    C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA
✓ 현재 위치:     C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\Korea_Market\analysis\한국기업_매출예측
✓ 운영체제:      nt



In [31]:
# ==================== 2. 필요한 모듈 import ====================
try:
    # 예측 함수 import (파일명 확인 필요!)
    from universal_ts_forecast_function import (
        forecast_one_from_pivot_inline,
        monitor_memory_usage
    )
    from stock_invest_function import fetch_table_data

    print("✓ 예측 모듈 import 성공")

except ImportError as e:
    print(f"❌ 모듈 import 실패: {e}")
    print("\n확인 사항:")
    print("1. DATA 폴더에 'universal_ts_forecast_function.py' 파일이 있는가?")
    print("2. DATA 폴더에 'stock_invest_function.py' 파일이 있는가?")
    print("\n파일명이 다르다면 위 import 문을 수정하세요.")
    sys.exit(1)

from DATA.stock_invest_function import *


def get_connection(db_info: dict):
    conn = pymysql.connect(
        host=db_info["host"],
        port=int(db_info["port"]),
        user=db_info["user"],
        password=db_info["password"],
        db=db_info.get("db", db_info.get("database")),
        charset="utf8mb4",
        cursorclass=pymysql.cursors.DictCursor,
    )
    return conn


def fetch_revenue_from_korea_fs_data(conn, ticker: str) -> pd.DataFrame:
    """
    korea_fs_data 에서 특정 ticker의 매출 관련 시계열을 가져오는 함수.
    ticker 는 '005930' 또는 'A005930' 둘 다 허용.
    """
    # ticker 앞에 A가 없으면 붙여주기
    if ticker.startswith("A"):
        symbol = ticker
    else:
        symbol = f"A{ticker}"

    sql = """
        SELECT `date`, `indicator`, `value`
        FROM korea_fs_data
        WHERE symbol = %s
          AND (
                indicator LIKE %s
             OR indicator LIKE %s
              )
        ORDER BY `date`
    """

    # LIKE 패턴은 파라미터로 넘겨서 % 에러 방지
    params = (symbol, "%매출%", "%수익%")

    df = pd.read_sql(sql, conn, params=params)

    if df.empty:
        print(f"⚠ 매출 관련 indicator가 발견되지 않았습니다. symbol={symbol}")
        return df

    df = df.rename(columns={"value": "revenue"})
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["revenue"] = pd.to_numeric(df["revenue"], errors="coerce")
    df = df.dropna(subset=["date"])

    return df[["date", "revenue", "indicator"]]



def fetch_revenue_from_dart_fs_data(conn, ticker: str) -> pd.DataFrame:
    sql = """
        SELECT
            report_date          AS date,
            thstrm_amount        AS revenue
        FROM korea_fs_data_from_DART
        WHERE ticker = %s
          AND account_nm IN ('수익(매출액)', '매출액')
        ORDER BY report_date
    """
    df = pd.read_sql(sql, conn, params=(ticker,))

    if not df.empty:
        df["date"] = df["date"].astype(str).str.strip()
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        df = df.dropna(subset=["date"])

        df["revenue"] = pd.to_numeric(df["revenue"], errors="coerce")
        df["source"] = "korea_fs_data_from_DART"

    return df


def get_merged_revenue_timeseries(db_info: dict, ticker: str) -> pd.DataFrame:
    conn = get_connection(db_info)
    try:
        df1 = fetch_revenue_from_korea_fs_data(conn, ticker)
        df2 = fetch_revenue_from_dart_fs_data(conn, ticker)

        frames = [df for df in [df1, df2] if not df.empty]
        if not frames:
            return pd.DataFrame(columns=["date", "revenue", "source"])

        merged = pd.concat(frames, ignore_index=True)
        merged = merged.sort_values("date").reset_index(drop=True)
        return merged[["date", "revenue", "source"]]
    finally:
        conn.close()

import pandas as pd

def extract_revenue_from_fs_df(fs_df: pd.DataFrame,
                               ticker: str,
                               keyword: str = "매출") -> pd.DataFrame:
    """
    korea_fs_data 전체 df(fs_df)에서 특정 ticker의 매출 관련 시계열만 뽑아오는 함수.

    Parameters
    ----------
    fs_df : pd.DataFrame
        fetch_table_data(db_info, "korea_fs_data") 로 가져온 전체 테이블
    ticker : str
        '005930' 또는 'A005930' 둘 다 허용
    keyword : str
        indicator 에 포함될 키워드 (기본값: '매출')

    Returns
    -------
    pd.DataFrame
        date, revenue, indicator 컬럼을 가진 시계열
    """

    # 1) 심볼 통일: 앞에 A 붙이기
    if ticker.startswith("A"):
        symbol = ticker
    else:
        symbol = "A" + ticker

    # 2) 해당 ticker + 매출 관련 indicator 필터링
    mask_symbol = fs_df["symbol"] == symbol
    mask_ind = fs_df["indicator"].astype(str).str.contains(keyword, na=False)

    sub = fs_df.loc[mask_symbol & mask_ind, ["date", "value", "indicator"]].copy()

    if sub.empty:
        print(f"⚠ {symbol} 에 대해 '{keyword}' 를 포함하는 indicator 가 없습니다.")
        print("   우선 아래 코드를 한 번 실행해서 indicator 목록을 눈으로 확인해 보세요:")
        print("   fs_df[fs_df['symbol']=='A005930']['indicator'].unique()")
        return sub

    # 3) 타입 정리
    sub["date"] = pd.to_datetime(sub["date"], errors="coerce")
    sub = sub.dropna(subset=["date"])

    sub["revenue"] = pd.to_numeric(sub["value"], errors="coerce")

    # 출력 형식 정리
    return sub[["date", "revenue", "indicator"]].sort_values("date").reset_index(drop=True)



# if __name__ == "__main__":
#     # 예시 db_info 채워 넣으신 뒤 테스트
#     db_info = {
#         "host": "192.168.0.230",
#         "port": 3307,
#         "user": "investar",
#         "password": "비밀번호입력",
#         "db": "investar",
#     }
#
#     ticker = "005930"
#     revenue_ts = get_merged_revenue_timeseries(db_info, ticker)
#     print(revenue_ts.head())
#     print(revenue_ts.tail())


✓ 예측 모듈 import 성공


In [32]:

# 예시 db_info 채워 넣으신 뒤 테스트
db_info = {
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'host': get_db_host(),  # 노트북에서는 다른 IP일 수 있음
    'port': 3307,
    'database': 'investar'
}

conn = get_connection(db_info)

# 이미 있으신 코드
fs_df = fetch_table_data(db_info, "korea_fs_data")
# 1) 권장: 숫자만 넣기
# revenue_df = extract_revenue_from_fs_df(fs_df, "005930", keyword="수익(매출액)")

✅ 'korea_fs_data' 테이블에서 5902708건의 데이터를 가져왔습니다.
⚠ A005930 에 대해 '수익(매출액)' 를 포함하는 indicator 가 없습니다.
   우선 아래 코드를 한 번 실행해서 indicator 목록을 눈으로 확인해 보세요:
   fs_df[fs_df['symbol']=='A005930']['indicator'].unique()


In [ ]:
from sqlalchemy import create_engine, text

In [48]:
revenue_df = extract_revenue_from_fs_df(fs_df, "005930", keyword="매출액(천원)")

⚠ A005930 에 대해 '매출액(천원) ' 를 포함하는 indicator 가 없습니다.
   우선 아래 코드를 한 번 실행해서 indicator 목록을 눈으로 확인해 보세요:
   fs_df[fs_df['symbol']=='A005930']['indicator'].unique()


In [49]:
revenue_df

,date,value,indicator


In [38]:
fs_df

,symbol,company_name,date,indicator,value
0,A000010,조흥은행,2004-01-31,수정PSR(연율화)(배),4.594400e-01
1,A000010,조흥은행,2004-02-29,수정PSR(연율화)(배),5.032000e-01
2,A000010,조흥은행,2004-03-31,계속사업이익(천원),3.612000e+07
3,A000010,조흥은행,2004-03-31,당기순이익(천원),3.612000e+07
4,A000010,조흥은행,2004-03-31,매출액(천원),7.920890e+08
...,...,...,...,...,...
5902703,A950220,네오이뮨텍,2025-06-30,지배주주지분(천원),2.374024e+07
5902704,A950220,네오이뮨텍,2025-06-30,지배주주총포괄이익(천원),-6.671558e+06
5902705,A950220,네오이뮨텍,2025-06-30,총자본(천원),2.374024e+07
5902706,A950220,네오이뮨텍,2025-06-30,총자산(천원),4.870483e+07


In [39]:
revenue_df

,date,value,indicator


In [59]:
import math
import pandas as pd
from sqlalchemy import create_engine, text

def create_fs_long_table_from_df(
    fs_df: pd.DataFrame,
    db_info: dict,
    table_name: str = "korea_fs_long_v2",
    chunksize: int = 20_000
):
    """
    fs_df(예: korea_fs_data 전체)를
    date, ticker, company_name, indicator, value 구조로 변환 후
    long-format 테이블로 저장.
    """
    # 1) 기본 정리
    print("🔎 원본 fs_df info")
    print(fs_df.info())
    print(fs_df.head())

    df = fs_df.copy()

    # symbol -> ticker (A005930 → 005930)
    df["ticker"] = df["symbol"].astype(str).str.replace("^A", "", regex=True)

    # 타입 정리
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["value"] = pd.to_numeric(df["value"], errors="coerce")

    # date 없는 행 제거
    before_drop = len(df)
    df = df.dropna(subset=["date"])
    print(f"✅ date not null: {before_drop:,} → {len(df):,}")

    # 최종 컬럼 순서
    df = df[["date", "ticker", "company_name", "indicator", "value"]]

    # (중요) 동일 (ticker, indicator, date) 중복 제거 – 마지막 값 기준으로
    df = df.sort_values(["ticker", "indicator", "date"])
    before_dup = len(df)
    df = df.drop_duplicates(subset=["ticker", "indicator", "date"], keep="last")
    print(f"✅ drop_duplicates: {before_dup:,} → {len(df):,}")

    print("🔎 정리된 df 샘플")
    print(df.head())

    if df.empty:
        print("❌ df 가 비어 있습니다. 업로드를 건너뜁니다.")
        return

    # 2) DB 엔진
    engine = create_engine(
        f"mysql+pymysql://{db_info['user']}:{db_info['password']}@"
        f"{db_info['host']}:{db_info['port']}/{db_info['database']}"
    )

    # 3) 테이블 스키마 생성 + TRUNCATE
    create_sql = f"""
    CREATE TABLE IF NOT EXISTS {table_name} (
        `date`         DATE            NOT NULL,
        `ticker`       VARCHAR(10)     NOT NULL,
        `company_name` VARCHAR(100)    NULL,
        `indicator`    VARCHAR(200)    NOT NULL,
        `value`        DECIMAL(20,4)   NULL,
        `created_at`   DATETIME        NOT NULL DEFAULT CURRENT_TIMESTAMP,
        `updated_at`   DATETIME        NOT NULL DEFAULT CURRENT_TIMESTAMP
                                      ON UPDATE CURRENT_TIMESTAMP,
        PRIMARY KEY (ticker, indicator, `date`),
        KEY idx_indicator_date (indicator, `date`),
        KEY idx_date (date)
    );
    """
    with engine.begin() as conn:
        conn.execute(text(create_sql))
        conn.execute(text(f"TRUNCATE TABLE {table_name};"))
    print(f"✅ {table_name} 스키마 생성 및 TRUNCATE 완료")

    # 4) chunk 단위 업로드
    n = len(df)
    n_chunks = math.ceil(n / chunksize)
    print(f"➡ 업로드 시작: 총 {n:,}행, chunksize={chunksize}, chunks={n_chunks}")

    for i in range(n_chunks):
        start = i * chunksize
        end = min((i + 1) * chunksize, n)
        chunk = df.iloc[start:end]

        print(f"   - chunk {i+1}/{n_chunks}: {start:,} ~ {end-1:,}  ({len(chunk):,} 행)")

        # 안정성 위해 method=None 으로 (multi가 불안하면)
        chunk.to_sql(
            name=table_name,
            con=engine,
            if_exists="append",
            index=False,
            method=None,
        )

    print("✅ 전체 업로드 완료")

    # 5) DB에 실제로 몇 행 들어갔는지 확인
    with engine.begin() as conn:
        result = conn.execute(text(f"SELECT COUNT(*) FROM {table_name};"))
        count = list(result)[0][0]
    print(f"📊 최종 DB row count: {count:,}")


In [60]:
create_fs_long_table_from_df(fs_df, db_info, table_name="korea_fs_data_from_DG", chunksize=20_000)


🔎 원본 fs_df info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5902708 entries, 0 to 5902707
Data columns (total 5 columns):
 #   Column        Dtype  
---  ------        -----  
 0   symbol        object 
 1   company_name  object 
 2   date          object 
 3   indicator     object 
 4   value         float64
dtypes: float64(1), object(4)
memory usage: 225.2+ MB
None
    symbol company_name        date      indicator         value
0  A000010         조흥은행  2004-01-31  수정PSR(연율화)(배)  4.594400e-01
1  A000010         조흥은행  2004-02-29  수정PSR(연율화)(배)  5.032000e-01
2  A000010         조흥은행  2004-03-31     계속사업이익(천원)  3.612000e+07
3  A000010         조흥은행  2004-03-31      당기순이익(천원)  3.612000e+07
4  A000010         조흥은행  2004-03-31        매출액(천원)  7.920890e+08
✅ date not null: 5,902,708 → 5,902,708
✅ drop_duplicates: 5,902,708 → 5,902,708
🔎 정리된 df 샘플
          date  ticker company_name   indicator     value
90  2005-03-31  000010         조흥은행  YoY_계속사업이익  2.485825
117 2005-06-30  000010    

In [61]:
import pandas as pd
from sqlalchemy import create_engine

# DB 엔진 생성 함수 (한 번만 정의해두고 계속 사용)
def get_engine(db_info: dict):
    engine = create_engine(
        f"mysql+pymysql://{db_info['user']}:{db_info['password']}@"
        f"{db_info['host']}:{db_info['port']}/{db_info['database']}"
    )
    return engine


def fetch_revenue_pivot(
    db_info: dict,
    ticker: str,
    table_name: str = "korea_fs_data_from_DG",
    indicator_name: str = "매출액(천원)",
) -> pd.DataFrame:
    """
    새 long 테이블에서 특정 ticker 의 '매출액(천원)' 시계열 pivot table 생성

    반환 형식 예:
                매출액(천원)
        date
        2004-03-31   7.9e+08
        2004-06-30   ...
        ...

    Parameters
    ----------
    db_info : dict
        host, port, user, password, database 정보를 담은 dict
    ticker : str
        '005930' 또는 'A005930' 둘 다 허용 (내부에서 정리)
    table_name : str
        long 테이블 이름 (기본: 'korea_fs_data_from_DG')
    indicator_name : str
        매출액 indicator 이름 (기본: '매출액(천원)')

    Returns
    -------
    pd.DataFrame
        index = date, column = indicator_name 인 pivot table
    """

    # ticker 형식 통일: A 붙어있으면 제거
    if ticker.startswith("A"):
        norm_ticker = ticker[1:]
    else:
        norm_ticker = ticker

    engine = get_engine(db_info)

    sql = f"""
        SELECT date, indicator, value
        FROM {table_name}
        WHERE ticker = %s
          AND indicator = %s
        ORDER BY date
    """

    df = pd.read_sql(sql, con=engine, params=(norm_ticker, indicator_name))

    if df.empty:
        print(f"⚠ ticker={norm_ticker}, indicator='{indicator_name}' 데이터가 없습니다.")
        return df

    # 타입 정리
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df = df.dropna(subset=["date"])

    # 피벗테이블: index=date, columns=indicator, values=value
    pivot = df.pivot_table(
        index="date",
        columns="indicator",
        values="value",
        aggfunc="first"
    ).sort_index()

    # columns 가 한 개라서 보기 좋게 column 이름만 정리 (선택)
    pivot.columns.name = None  # 'indicator' 헤더 제거

    return pivot


In [62]:
# 예: 삼성전자 매출액(천원) 피벗테이블
ticker = "005930"   # "A005930" 넣어도 됩니다

rev_pivot = fetch_revenue_pivot(db_info, ticker)

print(rev_pivot.head())
print(rev_pivot.tail())

                 매출액(천원)
date                    
2004-03-31  1.441363e+10
2004-06-30  1.497945e+10
2004-09-30  1.434394e+10
2004-12-31  1.389533e+10
2005-03-31  1.381218e+10
                 매출액(천원)
date                    
2024-06-30  7.406830e+10
2024-09-30  7.909873e+10
2024-12-31  7.578827e+10
2025-03-31  7.914050e+10
2025-06-30  7.456632e+10
